# Pose Detection Notebook — BlazePose (solutions API)

This notebook captures frames from your **webcam** and runs pose detection with:
- **BlazePose (MediaPipe solutions API)** — 33 keypoints, no TensorFlow dependency (recommended to start).


## 1) Install (only what you need)

```bash
# BlazePose (solutions API) — no TensorFlow required
pip install mediapipe==0.10.14 protobuf==4.25.3 opencv-python numpy`
```

In [5]:
# 2) Imports
import cv2, numpy as np
from collections import deque

In [6]:
# 3) Configuration
MODEL = "blazepose"   # choose: "blazepose"
SOURCE = 0            # 0 = webcam; or a path string like "video.mp4"
DRAW_SKELETON = True
SCORE_THRESH = 0.5
print(f"Backend selected: {MODEL}, source: {SOURCE}")

Backend selected: blazepose, source: 0


## 2) PoseDetector wrapper

- `infer(frame) -> list[(x, y, score)]` in pixel coords
- `draw(frame, pts)` overlays points + skeleton
- BlazePose path imports **only** `mediapipe.python.solutions.pose` (avoids pulling in TensorFlow).

In [7]:
class PoseDetector:
    def __init__(self, backend="blazepose"):
        backend = backend.lower()
        self.backend = backend
        if backend == "blazepose":
            self._init_blazepose()
        else:
            raise ValueError("backend must be 'blazepose'")

    # ---------- BlazePose (solutions API) ----------
    def _init_blazepose(self):
        # Import only the solutions submodule to avoid TensorFlow/tasks imports
        from mediapipe.python.solutions import pose as mp_pose
        self._mp_pose = mp_pose
        # model_complexity: 0=lite, 1=full, 2=heavy
        self.pose = mp_pose.Pose(
            model_complexity=1,
            enable_segmentation=False,
            smooth_landmarks=True
        )
        self._edges = list(mp_pose.POSE_CONNECTIONS)

    def _infer_blazepose(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        res = self.pose.process(img_rgb)
        if not res.pose_landmarks:
            return []
        pts = []
        for idx, lm in enumerate(res.pose_landmarks.landmark):  # 33 landmarks
            x = float(lm.x) * w
            y = float(lm.y) * h
            score = float(getattr(lm, "visibility", 0.9))
            pts.append((x, y, score))
        return pts

    # ---------- Unified API ----------
    def infer(self, frame_bgr):
        if self.backend == "blazepose":
            return self._infer_blazepose(frame_bgr)

    def draw(self, frame_bgr, pts, score_thresh=0.8):
        if not pts:
            return frame_bgr
        for (x, y, sc) in pts:
            if sc >= score_thresh:
                cv2.circle(frame_bgr, (int(x), int(y)), 3, (0,255,0), -1)
        if DRAW_SKELETON and self._edges:
            for a, b in self._edges:
                if a < len(pts) and b < len(pts):
                    if pts[a][2] >= score_thresh and pts[b][2] >= score_thresh:
                        ax, ay, _ = pts[a]; bx, by, _ = pts[b]
                        cv2.line(frame_bgr, (int(ax), int(ay)), (int(bx), int(by)), (0,255,0), 1)
        return frame_bgr

## 3) Utilities: movement & angles

In [8]:
def movement_magnitude(prev_pts, curr_pts):
    if not prev_pts or not curr_pts:
        return 0.0
    n = min(len(prev_pts), len(curr_pts))
    if n == 0: return 0.0
    disps = []
    for i in range(n):
        x1,y1,_ = prev_pts[i]
        x2,y2,_ = curr_pts[i]
        disps.append(((x2-x1)**2 + (y2-y1)**2) ** 0.5)
    return float(np.mean(disps)) if disps else 0.0

def angle_2pts(a, b):
    a = np.array(a[:2], float); b = np.array(b[:2], float)
    delta_y, delta_x = b[1]-a[1], b[0]-a[0]
    angle_rad = np.arctan2(delta_y, delta_x)
    angle_deg = np.degrees(angle_rad)
    return angle_deg

def angle_3pts(a, b, c):
    a = np.array(a[:2], float); b = np.array(b[:2], float); c = np.array(c[:2], float)
    ba, bc = a-b, c-b
    denom = (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-6)
    cosang = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0) # should check if it's possible to fall outside of the -1, 1 range
    return float(np.degrees(np.arccos(cosang)))

def torso_scale(pts):
    try:
        # BlazePose indices: shoulders 11/12, hips 23/24
        sL, sR, hL, hR = pts[11], pts[12], pts[23], pts[24]
        s_mid = ((sL[0]+sR[0])/2.0, (sL[1]+sR[1])/2.0)
        h_mid = ((hL[0]+hR[0])/2.0, (hL[1]+hR[1])/2.0)
        d = ((s_mid[0]-h_mid[0])**2 + (s_mid[1]-h_mid[1])**2)**0.5
        return max(d, 1.0)
    except:
        return 1.0

## 4) Quick sanity check (BlazePose solutions API)

In [9]:
from mediapipe.python.solutions import pose as mp_pose
p = mp_pose.Pose(); print('BlazePose solutions API imported OK')

BlazePose solutions API imported OK


## 5) Model Usage

In [10]:
ACTION = "shoulder_press"

In [11]:
def get_data_from_trainer_video(model_name, source=0):
    detector = PoseDetector(model_name)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print("Could not open video source, check path")
        return

    down = False
    reps = 0
    action = ACTION
    score = 100.0
    rep_min_angle = float("inf")
    max_asymmetry = 0
    trainer_data = []
    
    if action =="shoulder_press":
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            pts = detector.infer(frame)

            if pts:
                shoulder_right, elbow_right, wrist_right = pts[11], pts[13], pts[15]
                shoulder_left, elbow_left, wrist_left = pts[12], pts[14], pts[16]

                angle__right_elbow = angle_3pts(shoulder_right, elbow_right, wrist_right)
                if (angle__right_elbow is not None):
                        rep_min_angle = min(rep_min_angle, angle__right_elbow)

                angle_right_hand = angle_2pts(elbow_right, wrist_right)
                angle_left_hand = angle_2pts(elbow_left, wrist_left)

                symmetry_forearms = abs(angle_right_hand - angle_left_hand)
                if symmetry_forearms:
                     max_asymmetry = max(max_asymmetry, symmetry_forearms)
                     
    
        cap.release()
        trainer_data.append(rep_min_angle)
        trainer_data.append(max_asymmetry)
    
    return trainer_data

In [12]:
def run_demo(model_name, source=0, trainer_data = None):
    print(f"Starting with backend={model_name}, source={source}")
    detector = PoseDetector(model_name)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print("Could not open video source. Try SOURCE=0 for webcam or a valid file path.")
        return

    prev_pts = None
    mov_hist = deque(maxlen=30)
    LOW, HIGH = 70, 160  # squat-like thresholds
    down = False
    reps = 0
    action = ACTION # shoulder_press or squat
    score = 100.0
    

    # For shoulder_press
    if action == "shoulder_press":
        trainer_angle_degrees = trainer_data[0] + 10
        trainer_max_asymmetry = trainer_data[1] + 5
        rep_min_angle = float("inf")
        rep_min_angles = []
        rep_max_angle = float("-inf")
        rep_max_angles = []

        decrease_from_low_elbow_angle = 0.0
        decrease_from_lack_of_symmetry = 0.0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            pts = detector.infer(frame)

            move = movement_magnitude(prev_pts, pts)
            scale = torso_scale(pts) if pts else 1.0
            move_norm = move / scale
            mov_hist.append(move_norm)
            smooth = float(np.mean(mov_hist)) if mov_hist else 0.0
            prev_pts = pts

            if pts:
                shoulder_right, elbow_right, wrist_right = pts[11], pts[13], pts[15]
                shoulder_left, elbow_left, wrist_left = pts[12], pts[14], pts[16]

                angle_right_elbow = angle_3pts(shoulder_right, elbow_right, wrist_right)
                if angle_right_elbow:
                    cv2.putText(frame, f"Elbow angle: {angle_right_elbow:.1f} deg", (12, 84),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
                    rep_min_angle = min(rep_min_angle, angle_right_elbow)

                # Symmetry checkk
                angle_right_hand = angle_2pts(elbow_right, wrist_right)
                angle_left_hand = angle_2pts(elbow_left, wrist_left)

                symmetry_forearms = abs(angle_right_hand - angle_left_hand)

                if symmetry_forearms > trainer_max_asymmetry:
                    decrease_from_lack_of_symmetry += symmetry_forearms
                    score -= symmetry_forearms * 0.001 # small factor as it is applied on every frame

                # Rep counter and form scoring for end of rep               
                if angle_right_elbow:
                    if angle_right_elbow < trainer_angle_degrees + 13:
                        down = True
                    if down and angle_right_elbow > trainer_angle_degrees + 13:
                        reps += 1
                        down = False
                        if rep_min_angle != float("inf"):
                            decrease_from_low_elbow_angle += abs(rep_min_angle - trainer_angle_degrees)/abs(trainer_angle_degrees) * 10
                            score -= abs(rep_min_angle - trainer_angle_degrees)/abs(trainer_angle_degrees) * 10
                            rep_min_angles.append(rep_min_angle)
                        rep_min_angle = float("inf")
                
                vis = detector.draw(frame.copy(), pts, score_thresh=SCORE_THRESH)
                cv2.putText(vis, f"Reps: {reps}", (12, 112),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
                cv2.putText(vis, f"Current score: {score:.1f}", (12, 142),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 225, 200), 2)
                
            cv2.imshow("Pose (ESC to quit)", vis)
            if cv2.waitKey(1) & 0xFF == 27:
                break
        cap.release()
        cv2.destroyAllWindows()

        print(f"Total reps counted: {reps}")
        print(f"Final score: {max(score, 0):.1f}/100.0")

        if reps > 0:
            if decrease_from_low_elbow_angle > decrease_from_lack_of_symmetry:
                print("Improvement suggestion: go lower on the press to match trainer's form.")
            else:
                print("Improvement suggestion: keep your arms more symmetrical.")

    elif action == "squat":
        knee_angle = None
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            pts = detector.infer(frame)

            move = movement_magnitude(prev_pts, pts)
            scale = torso_scale(pts) if pts else 1.0
            move_norm = move / scale
            mov_hist.append(move_norm)
            smooth = float(np.mean(mov_hist)) if mov_hist else 0.0
            prev_pts = pts

            if pts:
                hip, knee, ankle = pts[23], pts[25], pts[27]

                knee_angle = angle_3pts(hip, knee, ankle)
                if knee_angle < LOW:
                    down = True
                if down and knee_angle is not None and knee_angle > HIGH:
                    reps += 1
                    down = False

            vis = detector.draw(frame.copy(), pts, score_thresh=SCORE_THRESH)
            cv2.putText(vis, f"Movement (norm): {smooth:.2f}", (12, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            if knee_angle is not None:
                cv2.putText(vis, f"Knee angle: {knee_angle:.1f} deg  Reps: {reps}",
                            (12, 56), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

            cv2.imshow("Pose (ESC to quit)", vis)
            if cv2.waitKey(1) & 0xFF == 27:
                break
        cap.release()
        cv2.destroyAllWindows()

        print(f"Total reps counted: {reps}")
        print(f"Final score: {max(score, 0):.1f}/100.0")
        if action == "shoulder_press":
            if reps > 0:
                avg_decrease = (decrease_from_low_elbow_angle + decrease_from_lack_of_symmetry) / reps
                print(f"Average score decrease per rep due to form: {avg_decrease:.1f} points")
                if decrease_from_low_elbow_angle > decrease_from_lack_of_symmetry:
                    print("Improvement suggestion: go lower on the press to match trainer's form.")
                else:
                    print("Improvement suggestion: work on keeping your arms more symmetrical.")

# "Train" the code on trainer video
source_train = "Videos\shoulder_press\shoulder_press_trainer.mp4"
trainer_data = get_data_from_trainer_video(MODEL, source_train)
if not trainer_data:
    print("Failed to get trainer data, exiting.")
    exit(1)

# Run it using the config values
source_test = "Videos\shoulder_press\shoulder_press_jimena.mp4"
run_demo(MODEL, source_test, trainer_data)

c:\Users\david\Desktop\Courses\2_Junior\CIS 4810\Final_Project\gAInz_cis5810\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Starting with backend=blazepose, source=Videos\shoulder_press\shoulder_press_jimena.mp4
Total reps counted: 1
Final score: 98.0/100.0
Improvement suggestion: go lower on the press to match trainer's form.
